In [3]:
from project import icu_query

/Users/jack/ucsf_courses_resources/winter 2026/datasci_223/final/venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [19]:
data = icu_query()


/Users/jack/ucsf_courses_resources/winter 2026/datasci_223/final/venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
original_query = data # to compare against as we make edits to data

In [21]:
data.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,admittime,dischtime,...,p_onset,p_end,qrs_onset,qrs_end,t_end,p_axis,qrs_axis,t_axis,ecg_time,report_0
0,12466550,23998182,30000153,Trauma SICU (TSICU),Trauma SICU (TSICU),2174-09-29 12:09:00,2174-10-01 03:26:10,1.636921,2174-09-29 10:43:00,2174-10-15 15:24:00,...,40,140,180,256,530,67,64,44,2174-09-29 14:56:00,Sinus rhythm.
1,18421337,22413411,30000484,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2136-01-14 17:23:32,2136-01-17 04:53:08,2.478889,2136-01-14 17:22:00,2136-01-24 16:00:00,...,40,132,268,374,680,45,79,33,2136-01-15 09:24:00,Sinus rhythm with 1st degree A-V block
2,12207593,22795209,30000646,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2194-04-29 01:39:22,2194-05-03 18:23:48,4.697523,2194-04-27 18:43:00,2194-05-06 02:29:00,...,40,162,214,308,556,66,7,28,2194-04-29 01:43:00,--- Warning: Data quality may affect interpret...
3,12980335,23552849,30001148,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2156-08-30 11:10:59,2156-08-31 14:25:34,1.135127,2156-08-29 11:45:00,2156-09-03 17:20:00,...,40,150,204,300,626,69,75,10,2156-08-30 15:20:00,Sinus bradycardia
4,12168737,29283664,30001336,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2186-03-20 00:44:48,2186-03-22 19:25:44,2.778426,2186-03-20 00:44:00,2186-03-22 19:10:00,...,40,166,212,330,604,54,33,-19,2186-03-20 00:46:00,Sinus rhythm.


In [26]:
# need to bucket report_0...
# getting frequency of report_0 codes
original_ecg_report0_frequency = original_query['report_0'].value_counts()
original_ecg_report0_frequency.to_csv('sanity_outputs/original_ecg_report0_frequency.csv')


In [ ]:
# sinus rhythm, sinus tachycardia, a.fib
data['report_0'] = data['report_0'].str.lower().str.removesuffix(".")
data = data[~data['report_0'].str.contains('warning')]
initial_drop = data['report_0'].value_counts()
initial_drop.to_csv('sanity_outputs/initial_drop.csv')
# handles: Sinus rhythm, Sinus tachycardia, Atrial fibrillation, Sinus bradycardia that had '.' suffix
# changes all to lower, drops records with 'warning' because those signals could harm our model




In [35]:
print(original_query.shape)
print(data.shape)

(35474, 33)
(34835, 33)


In [40]:
data_first_cats = data[data['report_0'].isin(['sinus rhythm', 'sinus tachycardia', 'atrial fibrillation', ' sinus bradycardia'])]
print(sum(data_first_cats['hospital_expire_flag']))
print(sum(data['hospital_expire_flag']))

1945
4195


In [41]:
1945/4195

0.4636471990464839

In [42]:
4195/34835

0.12042486005454285